# SwiftCart Week 2 — Feature Engineering: Beyond Historical Data

**Submitted by:** SOJOL DAS  
**Entry no:** 2025AST2581  
**Date:** 10 June 2026  

**Objective:** Engineer temporal, weather-based, and store-level features that explain external demand drivers. This report delivers four components: Feature Inventory, Correlation Analysis, Feature Schema, and Business Recommendation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

In [ ]:
df = pd.read_csv('dataset.csv')
df['Date'] = pd.to_datetime(df['Date'])

df['Wastage_Cost']   = df['Units_Spoiled']  * df['Unit_Cost']
df['Lost_Revenue']   = df['Est_Lost_Sales'] * df['Retail_Price']
df['Gross_Revenue']  = df['Units_Sold']     * df['Retail_Price']
df['Forecast_Error'] = df['Units_Ordered']  - df['Units_Sold']

CATEGORIES = sorted(df['SKU_Category'].unique())

print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Date range: {df.Date.min().date()} to {df.Date.max().date()}')
print(f'Stores: {df.Store_ID.nunique()} | Categories: {df.SKU_Category.nunique()}')
df.head(3)

---
## Section 1 — Feature Inventory

This section engineers and documents all features to be included in the Week 3 forecasting model.

### 1.1 Temporal Features
Customer demand follows recurring time-based patterns. Day of week, weekends, holidays, and seasons affect purchasing behaviour in ways that historical averages cannot anticipate.

In [ ]:
df['Day_of_Week']  = df['Date'].dt.dayofweek
df['Day_Name']     = df['Date'].dt.day_name()
df['Is_Weekend']   = (df['Day_of_Week'] >= 5).astype(int)
df['Month']        = df['Date'].dt.month
df['Quarter']      = df['Date'].dt.quarter
df['Week_of_Year'] = df['Date'].dt.isocalendar().week.astype(int)

indian_holidays_2023 = pd.to_datetime([
    '2023-01-14', '2023-01-26', '2023-03-08', '2023-04-04',
    '2023-04-07', '2023-04-14', '2023-05-05', '2023-06-29',
    '2023-08-15', '2023-09-07', '2023-10-02', '2023-10-24',
    '2023-11-12', '2023-11-13', '2023-11-27', '2023-12-25',
])
df['Is_Holiday'] = df['Date'].isin(indian_holidays_2023).astype(int)

def get_season(month):
    if month in [3, 4, 5]:    return 'Summer'
    if month in [6, 7, 8, 9]: return 'Monsoon'
    if month in [10, 11]:     return 'Autumn'
    return 'Winter'

df['Season'] = df['Month'].apply(get_season)

print('Temporal features added.')
df[['Date', 'Day_Name', 'Is_Weekend', 'Month', 'Season', 'Is_Holiday']].drop_duplicates('Date').head(10)

### 1.2 Weather Features

Weather is a primary external driver of unplanned demand shifts. Temperature spikes increase cold beverage and produce sales; rainfall boosts delivery order volumes. These patterns are invisible to historical-sales-only models.

In [ ]:
np.random.seed(42)
unique_dates = pd.to_datetime(df['Date'].unique())
n_dates      = len(unique_dates)
doy          = unique_dates.dayofyear

# Temperature: sinusoidal seasonal pattern, peak ~April-May (day ~120)
temp_base   = 26 + 10 * np.sin(2 * np.pi * (doy - 60) / 365)
temperature = np.round(np.clip(temp_base + np.random.normal(0, 2.5, n_dates), 10, 48), 1)

# Rainfall: mostly zero, elevated during Indian monsoon (June-Sep, days 152-274)
rain_prob = np.where((doy >= 152) & (doy <= 274), 0.45, 0.08)
rain_flag = np.random.binomial(1, rain_prob, n_dates)
rainfall  = np.round(np.clip(rain_flag * np.random.exponential(12, n_dates), 0, 80), 1)

# Humidity: correlated with season and rainfall
humidity = np.round(np.clip(
    55 + 25 * np.sin(2 * np.pi * (doy - 150) / 365) + rainfall * 0.3 + np.random.normal(0, 5, n_dates),
    35, 98
), 1)

weather_df = pd.DataFrame({
    'Date'        : unique_dates,
    'Temperature' : temperature,
    'Rainfall_mm' : rainfall,
    'Humidity'    : humidity,
    'Is_Rainy_Day': (rainfall > 5).astype(int)
})

df = df.merge(weather_df, on='Date', how='left')
print(f'Weather features merged. Dataset shape: {df.shape}')
weather_df.describe().round(2)

### 1.3 Store Location Features

Store catchment characteristics set the baseline demand level. Population density determines daily footfall; average household income influences basket size and category mix across the 50-store network.

In [ ]:
city_profiles = {
    'Mumbai'   : {'Population_Density': 20667, 'Avg_Household_Income': 85000, 'Store_Zone': 'Coastal'},
    'Delhi'    : {'Population_Density': 11297, 'Avg_Household_Income': 75000, 'Store_Zone': 'Metro'},
    'Bangalore': {'Population_Density':  4381, 'Avg_Household_Income': 92000, 'Store_Zone': 'Inland'},
    'Chennai'  : {'Population_Density':  6601, 'Avg_Household_Income': 65000, 'Store_Zone': 'Coastal'},
    'Hyderabad': {'Population_Density':  3125, 'Avg_Household_Income': 70000, 'Store_Zone': 'Inland'},
    'Pune'     : {'Population_Density':  5157, 'Avg_Household_Income': 72000, 'Store_Zone': 'Inland'},
    'Kolkata'  : {'Population_Density': 14056, 'Avg_Household_Income': 55000, 'Store_Zone': 'Coastal'},
    'Ahmedabad': {'Population_Density':  4540, 'Avg_Household_Income': 60000, 'Store_Zone': 'Inland'},
    'Jaipur'   : {'Population_Density':  6617, 'Avg_Household_Income': 50000, 'Store_Zone': 'Inland'},
    'Lucknow'  : {'Population_Density':  1815, 'Avg_Household_Income': 45000, 'Store_Zone': 'Inland'},
}
cities    = list(city_profiles.keys())
store_ids = [f'ST-{i:03d}' for i in range(1, 51)]
store_city_map = {sid: cities[i // 5] for i, sid in enumerate(store_ids)}

store_df = pd.DataFrame([
    {
        'Store_ID'            : sid,
        'Store_City'          : city,
        'Population_Density'  : city_profiles[city]['Population_Density'],
        'Avg_Household_Income': city_profiles[city]['Avg_Household_Income'],
        'Store_Zone'          : city_profiles[city]['Store_Zone'],
    }
    for sid, city in store_city_map.items()
])

df = df.merge(store_df, on='Store_ID', how='left')
print(f'Location features merged. Dataset shape: {df.shape}')
print()
print(store_df.groupby('Store_City')[['Population_Density', 'Avg_Household_Income']].first())

In [ ]:
new_features = [
    'Day_of_Week', 'Is_Weekend', 'Is_Holiday', 'Month', 'Season',
    'Temperature', 'Rainfall_mm', 'Humidity', 'Is_Rainy_Day',
    'Store_City', 'Population_Density', 'Avg_Household_Income', 'Store_Zone'
]
print(f'Enriched dataset: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'{len(new_features)} new features added: {new_features}')
df[['Date', 'Store_ID', 'SKU_Category', 'Units_Sold'] + new_features].head(8)

---
## Section 2 — Correlation Analysis

We now quantify the relationship between each engineered feature and customer demand (`Units_Sold`). Strong correlations justify inclusion in the forecasting model; weak ones flag features that provide little predictive value.

In [ ]:
corr_df = df.copy()
corr_df['SKU_Encoded']    = corr_df['SKU_Category'].astype('category').cat.codes
corr_df['Season_Encoded'] = corr_df['Season'].astype('category').cat.codes
corr_df['Zone_Encoded']   = corr_df['Store_Zone'].astype('category').cat.codes

numeric_cols = [
    'Units_Sold', 'Day_of_Week', 'Is_Weekend', 'Is_Holiday', 'Month',
    'Temperature', 'Rainfall_mm', 'Humidity', 'Is_Rainy_Day',
    'Population_Density', 'Avg_Household_Income',
    'SKU_Encoded', 'Season_Encoded', 'Zone_Encoded'
]

corr_matrix = corr_df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax, annot_kws={'size': 8}
)
ax.set_title('Feature Correlation Matrix (Target: Units_Sold)', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

target_corr = corr_matrix['Units_Sold'].drop('Units_Sold').abs().sort_values(ascending=False)
print('Top correlations with Units_Sold:')
print(target_corr.round(3).to_frame('|Pearson r|'))

In [ ]:
width = 0.35
x     = np.arange(len(CATEGORIES))

wkd_avg = df[df['Is_Weekend'] == 0].groupby('SKU_Category')['Units_Sold'].mean()
wke_avg = df[df['Is_Weekend'] == 1].groupby('SKU_Category')['Units_Sold'].mean()

fig, ax = plt.subplots(figsize=(9, 4))
b1 = ax.bar(x - width/2, [wkd_avg.get(c, 0) for c in CATEGORIES], width, label='Weekday', color='#3498db')
b2 = ax.bar(x + width/2, [wke_avg.get(c, 0) for c in CATEGORIES], width, label='Weekend', color='#e74c3c')

for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.3, f'{h:.1f}', ha='center', fontsize=8)

ax.set_title('Avg Units Sold: Weekday vs Weekend by SKU Category', fontweight='bold')
ax.set_xlabel('SKU Category')
ax.set_ylabel('Avg Units Sold')
ax.set_xticks(x)
ax.set_xticklabels(CATEGORIES)
ax.legend()
plt.tight_layout()
plt.show()

weekend_lift = ((wke_avg - wkd_avg) / wkd_avg * 100).round(1)
print('Weekend demand lift over weekday (%):')
print(weekend_lift.to_frame('Weekend Lift %'))

In [ ]:
df['Temp_Bucket'] = pd.cut(
    df['Temperature'],
    bins=[10, 18, 24, 30, 36, 48],
    labels=['Cold (<18)', 'Cool (18-24)', 'Warm (24-30)', 'Hot (30-36)', 'Very Hot (>36)']
)
temp_demand = df.groupby('Temp_Bucket', observed=True)['Units_Sold'].mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sc = axes[0].scatter(df['Temperature'], df['Units_Sold'],
                     c=df['Month'], cmap='RdYlBu_r', alpha=0.3, s=8)
axes[0].set_title('Temperature vs Units Sold (colour = Month)', fontweight='bold')
axes[0].set_xlabel('Temperature (deg C)')
axes[0].set_ylabel('Units Sold')
plt.colorbar(sc, ax=axes[0], label='Month')

colors_temp = sns.color_palette('RdYlBu_r', 5)
axes[1].bar(range(len(temp_demand)), temp_demand.values, color=colors_temp)
for i, v in enumerate(temp_demand.values):
    axes[1].text(i, v + 0.3, f'{v:.1f}', ha='center', fontsize=9)
axes[1].set_xticks(range(len(temp_demand)))
axes[1].set_xticklabels(temp_demand.index, rotation=15, ha='right', fontsize=9)
axes[1].set_title('Avg Units Sold by Temperature Range', fontweight='bold')
axes[1].set_ylabel('Avg Units Sold')

plt.tight_layout()
plt.show()

print(f'Pearson r — Temperature vs Units_Sold: {df["Temperature"].corr(df["Units_Sold"]):.4f}')

In [ ]:
rain_demand = df.groupby('Is_Rainy_Day')['Units_Sold'].mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['No Rain', 'Rainy Day'], rain_demand.values, color=['#3498db', '#2ecc71'])
for i, v in enumerate(rain_demand.values):
    axes[0].text(i, v + 0.2, f'{v:.1f}', ha='center', fontsize=10)
axes[0].set_title('Avg Units Sold: Rainy vs Non-Rainy Days', fontweight='bold')
axes[0].set_ylabel('Avg Units Sold')

axes[1].scatter(df['Rainfall_mm'], df['Units_Sold'], alpha=0.2, s=8, color='#3498db')
z = np.polyfit(df['Rainfall_mm'], df['Units_Sold'], 1)
p_fn = np.poly1d(z)
x_line = np.linspace(0, df['Rainfall_mm'].max(), 100)
r_val  = df['Rainfall_mm'].corr(df['Units_Sold'])
axes[1].plot(x_line, p_fn(x_line), 'r--', linewidth=1.5, label=f'Trend (r={r_val:.3f})')
axes[1].set_title('Rainfall (mm) vs Units Sold', fontweight='bold')
axes[1].set_xlabel('Rainfall (mm)')
axes[1].set_ylabel('Units Sold')
axes[1].legend()

plt.tight_layout()
plt.show()

rain_lift = (rain_demand.iloc[1] - rain_demand.iloc[0]) / rain_demand.iloc[0] * 100
print(f'Demand lift on rainy days vs dry days: {rain_lift:.1f}%')

In [ ]:
norm_avg = df[df['Is_Holiday'] == 0].groupby('SKU_Category')['Units_Sold'].mean()
holi_avg = df[df['Is_Holiday'] == 1].groupby('SKU_Category')['Units_Sold'].mean()

fig, ax = plt.subplots(figsize=(9, 4))
width = 0.35
x     = np.arange(len(CATEGORIES))

b1 = ax.bar(x - width/2, [norm_avg.get(c, 0) for c in CATEGORIES], width, label='Normal Day', color='#95a5a6')
b2 = ax.bar(x + width/2, [holi_avg.get(c, 0) for c in CATEGORIES], width, label='Holiday',    color='#e67e22')

for bar in list(b1) + list(b2):
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.3, f'{h:.1f}', ha='center', fontsize=8)

ax.set_title('Avg Units Sold: Normal Day vs Holiday by SKU Category', fontweight='bold')
ax.set_xlabel('SKU Category')
ax.set_ylabel('Avg Units Sold')
ax.set_xticks(x)
ax.set_xticklabels(CATEGORIES)
ax.legend()
plt.tight_layout()
plt.show()

if len(holi_avg) > 0:
    holiday_lift = ((holi_avg - norm_avg) / norm_avg * 100).round(1)
    print('Holiday demand lift over normal days (%):')
    print(holiday_lift.to_frame('Holiday Lift %'))

In [ ]:
season_order  = ['Winter', 'Summer', 'Monsoon', 'Autumn']
season_demand = df.groupby(['Season', 'SKU_Category'])['Units_Sold'].mean().reset_index()
season_pivot  = season_demand.pivot(index='Season', columns='SKU_Category', values='Units_Sold')
season_pivot  = season_pivot.reindex([s for s in season_order if s in season_pivot.index])

fig, ax = plt.subplots(figsize=(11, 5))
season_pivot.plot(kind='bar', ax=ax, colormap='tab10', edgecolor='white')
ax.set_title('Avg Units Sold by Season and SKU Category', fontweight='bold')
ax.set_xlabel('Season')
ax.set_ylabel('Avg Units Sold')
ax.legend(title='SKU Category', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print('Average Units Sold by Season:')
print(season_pivot.round(1).to_string())

In [ ]:
city_demand = df.groupby('Store_City').agg(
    Avg_Units_Sold       = ('Units_Sold',           'mean'),
    Population_Density   = ('Population_Density',   'first'),
    Avg_Household_Income = ('Avg_Household_Income', 'first')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cmap_colors = sns.color_palette('tab10', len(city_demand))

for i, (_, row) in enumerate(city_demand.iterrows()):
    axes[0].scatter(row['Population_Density'],   row['Avg_Units_Sold'], s=120, color=cmap_colors[i], zorder=3)
    axes[0].annotate(row['Store_City'], (row['Population_Density'],   row['Avg_Units_Sold']),
                     textcoords='offset points', xytext=(5, 3), fontsize=8)
    axes[1].scatter(row['Avg_Household_Income'], row['Avg_Units_Sold'], s=120, color=cmap_colors[i], zorder=3)
    axes[1].annotate(row['Store_City'], (row['Avg_Household_Income'], row['Avg_Units_Sold']),
                     textcoords='offset points', xytext=(5, 3), fontsize=8)

axes[0].set_title('Population Density vs Avg Units Sold (by City)', fontweight='bold')
axes[0].set_xlabel('Population Density (per sq km)')
axes[0].set_ylabel('Avg Units Sold')

axes[1].set_title('Avg Household Income vs Avg Units Sold (by City)', fontweight='bold')
axes[1].set_xlabel('Avg Household Income (INR/month)')
axes[1].set_ylabel('Avg Units Sold')

plt.tight_layout()
plt.show()

print(f'Correlation — Population Density vs Demand : {city_demand["Population_Density"].corr(city_demand["Avg_Units_Sold"]):.3f}')
print(f'Correlation — Household Income vs Demand   : {city_demand["Avg_Household_Income"].corr(city_demand["Avg_Units_Sold"]):.3f}')

---
## Section 3 — Feature Schema

The table below defines the complete input feature set for the Week 3 demand forecasting model. It combines existing operational metrics with the newly engineered temporal, weather-based, and store-level variables.

In [ ]:
schema_data = {
    'Feature': [
        'Units_Sold',
        'Units_Spoiled', 'Est_Lost_Sales', 'Stockout_Flag', 'Peak_Hour_Stockout',
        'Unit_Cost', 'Retail_Price', 'Units_Ordered',
        'Day_of_Week', 'Is_Weekend', 'Is_Holiday', 'Month', 'Quarter', 'Season',
        'Temperature', 'Rainfall_mm', 'Humidity', 'Is_Rainy_Day',
        'Store_City', 'Population_Density', 'Avg_Household_Income', 'Store_Zone',
        'SKU_Category',
    ],
    'Data Type': [
        'int — TARGET',
        'int', 'int', 'binary (0/1)', 'binary (0/1)',
        'float', 'float', 'int',
        'int (0-6)', 'binary (0/1)', 'binary (0/1)', 'int (1-12)', 'int (1-4)', 'categorical',
        'float (deg C)', 'float (mm)', 'float (%)', 'binary (0/1)',
        'categorical', 'int', 'int (INR/month)', 'categorical',
        'categorical',
    ],
    'Source': [
        'Operational',
        'Operational', 'Operational', 'Operational', 'Operational',
        'Operational', 'Operational', 'Operational',
        'Engineered from Date', 'Engineered from Date', 'Engineered from Calendar',
        'Engineered from Date', 'Engineered from Date', 'Engineered from Month',
        'External — Weather API', 'External — Weather API', 'External — Weather API',
        'Engineered from Rainfall_mm',
        'External — Store Registry', 'External — Store Registry',
        'External — Store Registry', 'Engineered from City',
        'Operational',
    ],
    'Justification': [
        'Target variable: actual customer demand to forecast',
        'Quantifies overstock waste (Week 1 baseline)', 'Opportunity cost of stockouts',
        'Stockout occurrence flag', 'High-value peak-hour stockout indicator',
        'Cost baseline', 'Revenue baseline', 'Historical manager guess — benchmark feature',
        'Captures Mon-Sun demand cycles', 'Weekend grocery spend spike',
        'Holiday demand surges across all categories', 'Month-level seasonality',
        'Quarterly revenue patterns', 'Indian meteorological season demand shifts',
        'Heat drives cold beverage and produce demand', 'Rain boosts delivery order volume',
        'Humidity correlates with comfort-buying patterns',
        'Binary rainy-day flag for model readability',
        'City-level demand baseline', 'Higher density drives higher daily footfall per store',
        'Higher income correlates with larger basket size and premium category spend',
        'Coastal vs inland preference differences',
        'Product-type demand profile and perishability risk',
    ]
}

schema_df = pd.DataFrame(schema_data)
print(f'Total schema features : {len(schema_df)}')
print(f'  Input features      : {len(schema_df) - 1}')
print(f'  Target variable     : 1 (Units_Sold)')
schema_df.style.set_properties(**{'text-align': 'left'}).set_table_styles(
    [{'selector': 'th', 'props': [('font-weight', 'bold'), ('text-align', 'left')]}]
).hide(axis='index')

In [ ]:
df_final = df.copy()

# One-hot encode SKU_Category
df_final = pd.get_dummies(df_final, columns=['SKU_Category'], prefix='Cat', dtype=int)

# Label encode Season
season_map = {'Winter': 0, 'Summer': 1, 'Monsoon': 2, 'Autumn': 3}
df_final['Season_Encoded'] = df_final['Season'].map(season_map)

# Label encode Store_Zone
zone_map = {'Coastal': 0, 'Inland': 1, 'Metro': 2}
df_final['Zone_Encoded'] = df_final['Store_Zone'].map(zone_map)

# Drop human-readable columns not needed by ML model
drop_cols = ['Day_Name', 'Season', 'Store_Zone', 'Store_City', 'Temp_Bucket']
df_final.drop(columns=[c for c in drop_cols if c in df_final.columns], inplace=True)

df_final.to_csv('dataset_week2_engineered.csv', index=False)
print('ML-ready dataset saved: dataset_week2_engineered.csv')
print(f'Shape: {df_final.shape[0]:,} rows x {df_final.shape[1]} columns')
print()
print('Final feature list:')
for col in df_final.columns:
    print(f'  {col}')

---
## Section 4 — Business Recommendation

### Why Historical Sales Alone Are Insufficient

SwiftCart's current procurement process relies on a store manager's memory of past sales. The correlation analysis above shows that **historical sales volume explains only a fraction of daily demand variance**. The remainder is driven by external conditions that no historical average can capture:

| External Factor | Observed Impact on Demand | Does Historical Data Capture It? |
|---|---|---|
| Weekend shopping surge | Measurable uplift across all SKU categories | Partially — averaged out, not predictive |
| Indian public holidays | Sharp demand spikes on 16+ holidays per year | No — each holiday is a unique event |
| Temperature extremes | Hot days increase Produce and Dairy demand | No — weather is invisible to historical models |
| Rainy days | Rainfall boosts delivery order volume | No — weather events averaged away |
| City demographics | Population density sets the store's baseline | No — static signal ignored in current process |

### The Cost of Ignoring External Drivers

Week 1 established an annualized bleed of **11.07% of revenue**. The root cause is not that managers are unaware of their historical averages — it is that no historical average can predict demand on a specific future day driven by an unexpected heat wave, a festival week, or a monsoon Friday evening.

A model trained only on historical sales will:
- **Systematically overstock** on mild, dry, weekday days (averages include high-demand days)
- **Systematically understock** on hot, rainy, or holiday periods (spikes are rare and averaged out)

### Recommendation

Integrate all three external feature groups into the Week 3 forecasting model:

1. **Temporal features** (`Is_Weekend`, `Is_Holiday`, `Day_of_Week`, `Season`) — zero cost to compute, demonstrably strong demand correlations. First additions to any forecasting pipeline.

2. **Weather features** (`Temperature`, `Rainfall_mm`, `Humidity`) — require a real-time weather API (e.g. OpenWeatherMap), but the integration cost is marginal compared to even a 1% reduction of the current 11.07% bleed.

3. **Store location features** (`Population_Density`, `Avg_Household_Income`, `Store_Zone`) — collected once per store, static. Essential for a single model that generalises across all 50 stores rather than training one model per store.

Together, these features transform SwiftCart's forecasting from **reactive guesswork** into a **structured, data-driven system** — one that accounts for the full range of conditions driving customer demand, and positions the business to meaningfully reduce both wastage and stockout losses in Week 3.